# Eleven-city segregation analysis: 2020 city-limit tract portions

Philadelphia, Miami, Los Angeles, Salt Lake City, San Francisco, Washington DC,
Houston, Detroit, Chicago, Phoenix and Atlanta. Four overlapping categories:
White alone, Black alone, Hispanic/Latino any race, Asian alone.

The analysis uses **individual populated Census tract portions inside 2020 city limits**.
Counts are summed from city-assigned Census blocks and checked against official place totals.
No racial counts are allocated by area. Zero-population portions are excluded from matrix
normalization. The resulting 4,738 nodes are not comparable in scale to NYC's earlier 70 groups.

All cells below read saved local outputs. **No Routes API calls are made.**


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/multicity_analysis.py").exists())
BASE = ROOT / "outputs/multicity"
comparison = pd.read_csv(BASE / "city_comparison.csv")
coverage = pd.read_csv(BASE / "city_coverage.csv")
print("Cities:", comparison.city.nunique(), "City/category results:", len(comparison))
print("Populated tract portions:", comparison.groupby("city").populated_tract_portions.first().sum())
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(comparison[["city", "group", "population", "population_share_K", "segregation_index",
        "segregation_index_normalized", "fryer_majority_ssi", "fryer_majority_ssi_minus_K",
        "fryer_majority_ssi_normalized", "majority_zone_count", "network_isolation", "dissimilarity_vs_rest"]])


## Travel model and comparison limits

Public rail and bus routes come from saved agency-produced GTFS feeds. Travel is an
**expected-time approximation** using actual stop sequences, median scheduled ride/dwell
times and half the observed mean headway in a morning service window. It is not an exact
8 a.m. itinerary or a real-time estimate. Read each feed's chosen service date below;
feeds may have different validity periods. Walking uses distance × 1.3 at 4.8 km/hour.

Water polygons conservatively screen transit access/egress/transfers and direct walking;
straight-line checks can reject legitimate bridge or indirect land paths. The primary
cost matrix uses those walking candidates and actual GTFS transit candidates. Only
pairs missing both receive a **flagged distance-only walking fallback**, which can cross
water and should not be interpreted as a verified pedestrian route. The incomplete
barrier-screened cost matrix is also saved separately.

Shared illustrative parameters: beta=0.1 per dollar, value of time=$15/hour, local floor=$1.25,
and **standardized $3 per transit journey**, including transfers. This is a comparison scenario,
not city-specific fare pricing. Public transit includes rail/bus only; no taxi, driving or invented
transit fallback is used.


In [ ]:
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(coverage[["city", "tract_count", "population", "active_bus_routes", "active_rail_routes",
        "service_dates", "same_date_across_city_feeds", "transit_candidate_pairs", "nonlocal_pairs",
        "distance_walk_fallback_pairs", "distance_walk_fallback_percent", "tract_portions_under_100_people"]])
print("Sources:", BASE / "transit_sources.json")


## PSSI, Fryer and tract-level eigenvectors

$P_{ij}=F_{ij}D_j/\sum_k F_{ik}D_k$, $A_r=P\operatorname{diag}(p_r)$.
There is **one dominant PSSI eigenvalue per city/category**. Its right eigenvector gives
one entry per tract, normalized to sum to one. We also export $n v_i$ (city mean=1) to
compare relative tract scores across cities with different tract counts.

Fryer uses the notebook's majority-tract adaptation ($p_i\geq0.5$) on the same $P$.
Each majority component has its own eigenvalue $\lambda_C$ and eigenvector $v_C$.
Its tract SSI is $s_i=|C|\lambda_C v_{C,i}$; the mean over all majority tracts equals the
city Fryer SSI. Nonmajority tracts have blank local Fryer scores. Local SSI may exceed one.

For each category, $K=N_r/N$:
- Normalized PSSI: $(\lambda-K)/(1-K)$.
- Requested adjusted Fryer: $SSI-K$.
- Additional rescaled Fryer: $(SSI-K)/(1-K)$.

Negative adjustments are retained. When no tract is majority-category, the notebook's
Fryer score is zero by convention; this does not imply absence of segregation among its
residents. The [paper](https://eml.berkeley.edu/~fechenique/published/echenique-fryer-qje.pdf)
subtracts a simulated baseline for its adjusted residential SSI; that simulation is not
computed here. Tiny city-boundary tract portions especially affect majority-tract counts
(Houston has 170 with fewer than 100 people).


In [ ]:
CITY = "philadelphia"  # Change to any city_id in comparison.city_id.unique().
GROUP = "black_alone"  # white_alone, black_alone, hispanic_latino_any_race, asian_alone
folder = BASE / CITY
tracts = pd.read_csv(folder / "analysis/tract_scores.csv", dtype={"tract_geoid": str})
selected = tracts.loc[tracts.group.eq(GROUP)].sort_values("matrix_index")
display(selected.sort_values("pssi_right_eigenvector", ascending=False).head(20))
with np.load(folder / "analysis/analysis_matrices.npz", allow_pickle=False) as saved:
    r = saved["group_names"].tolist().index(GROUP)
    A, v, eigenvalue = saved["A"][r], saved["right_eigenvectors"][:, r], saved["eigenvalues"][r]
    assert saved["tract_geoid"].tolist() == selected.tract_geoid.tolist()
    assert np.isclose(v.sum(), 1)
    assert np.max(np.abs(A @ v - eigenvalue * v)) < 1e-9
    display(pd.DataFrame(A[:5, :5], index=selected.tract_geoid.iloc[:5], columns=selected.tract_geoid.iloc[:5]))
print("City/category eigenvalue:", eigenvalue)
majority = selected[selected.fryer_is_majority_tract]
if len(majority):
    assert np.isclose(majority.fryer_tract_ssi.mean(), selected.fryer_city_ssi.iloc[0])
    display(majority.sort_values("fryer_tract_ssi", ascending=False).head(20))
else:
    print("No majority tracts for this category; local Fryer scores are not applicable.")


In [ ]:
# Optional local tract map: uses Census polygons, no external map tiles.
import geopandas as gpd
import matplotlib.pyplot as plt
geometry = gpd.read_file(folder / "demographics/nodes.geojson")
geometry["tract_geoid"] = geometry["tract_geoid"].astype(str)
map_data = geometry.merge(selected[["tract_geoid", "pssi_eigenvector_times_n", "fryer_tract_ssi"]], on="tract_geoid", validate="one_to_one")
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
map_data.plot(column="pssi_eigenvector_times_n", ax=axes[0], legend=True, cmap="viridis", linewidth=.15, edgecolor="white")
axes[0].set_title(f"{CITY}: {GROUP} — PSSI n × v (mean 1)")
if map_data.fryer_tract_ssi.notna().any():
    map_data.plot(column="fryer_tract_ssi", ax=axes[1], legend=True, cmap="magma", linewidth=.15, edgecolor="white", missing_kwds={"color":"lightgray"})
else:
    map_data.plot(ax=axes[1], color="lightgray", linewidth=.15, edgecolor="white")
axes[1].set_title("Fryer tract SSI; gray = nonmajority")
for ax in axes: ax.set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
# Full definitions for both output tables.
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(pd.read_csv(BASE / "score_column_dictionary.csv"))
    display(pd.read_csv(BASE / "tract_score_dictionary.csv"))
print("Comparison report:", BASE / "city_comparison.html")
